# Example 1.4.1 / 2: Keeping a Formula Linear

Companion notebook for **1.4.1 Linear Expressions and sum** in the AMPL lab Documentation, section "Keeping a Formula Linear". It runs the page's example from top to bottom: the model, the amplpy way (data.py, run.py), then the pure-AMPL way (data.dat, model.run). Solver: HiGHS.

## Install Dependencies
💡 TIP: Visit [portal.ampl.com](https://portal.ampl.com/user/ampl/license/list) or [dev.ampl.com](https://dev.ampl.com/ampl/install.html) for AMPL setup on Colab, Docker, Python, Windows, Linux, macOS

In [ ]:
# Install dependencies
%pip install -q amplpy

# Google Colab & Kaggle integration
from amplpy import AMPL, ampl_notebook
ampl = ampl_notebook(
    modules=["highs"],      # modules to install
    license_uuid="default") # license to use

## Keeping a Formula Linear

When total production is strictly positive, multiplying both sides by the total gives the same
rule, now linear. The linear form also remains well-defined when every `Make[p]` is zero, whereas
the ratio is then undefined. `expand` on one member,
`Limit["bands"]`, shows that AMPL has moved every variable to the left and collected the
coefficients. A scalar parameter such as `max_frac` takes a number in `data.py`:

### model.mod
The model, written to a file that both ways of running it read.

In [2]:
%%writefile model.mod
set PROD;                   # products
param max_frac > 0, <= 1;   # largest share of total production for one product
var Make {PROD} >= 0;       # tons made
# the same share limit, multiplied by the total
subject to Limit {p in PROD}:
    Make[p] <= max_frac * sum {q in PROD} Make[q];     # linear

Writing model.mod


### amplpy: data.py and run.py

`ampl` is the `AMPL` object created above. Each run starts from a fresh session that has read `model.mod`; then `data.py` sets the data from Python and `run.py` solves and reads the results.

In [3]:
# A fresh session that has read model.mod, as data.py and run.py assume
ampl.reset()
ampl.eval("reset options;")  # reset keeps the options
ampl.read("model.mod")

In [4]:
# data.py
ampl.set["PROD"] = ["bands", "coils", "plate"]
ampl.param["max_frac"] = 0.4

In [5]:
# run.py
print(ampl.get_output('expand Limit["bands"];'))

subject to Limit['bands']:
	0.6*Make['bands'] - 0.4*Make['coils'] - 0.4*Make['plate'] <= 0;




### Pure AMPL: data.dat and model.run

Pure AMPL:

In [6]:
%%writefile data.dat
set PROD := bands coils plate;
param max_frac := 0.4;

Writing data.dat


In [7]:
%%ampl_eval
reset;
reset options;
model model.mod;
data data.dat;

In [8]:
%%ampl_eval
# model.run
expand Limit["bands"];

subject to Limit['bands']:
	0.6*Make['bands'] - 0.4*Make['coils'] - 0.4*Make['plate'] <= 0;

